In [8]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import tomllib
import gradio as gr

In [ ]:
# --- CONFIG
MODELS_DIR = "./models"
TOML_PATH = "../tp-dataset/categories/typst.toml"
IMG_SIZE = 28

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# glyphs
with open(TOML_PATH, "rb") as f:
    data = tomllib.load(f)
GLYPHS = [filename.replace(".toml", "") for filename in data.get("wordlist", {}).get("list", [])]

Using device: mps


In [11]:
# saved weights
class SitelenPonaCNN(nn.Module):
    def __init__(self, num_classes=137):
        super(SitelenPonaCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) 
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [12]:
# --- 3. LOAD ALL MODELS ---
# We will store them in a dictionary: {"model_filename": loaded_model_object}
loaded_models = {}

if not os.path.exists(MODELS_DIR):
    os.makedirs(MODELS_DIR)
    print(f"Created {MODELS_DIR}. Please drop your .pth files here!")

model_files = glob.glob(os.path.join(MODELS_DIR, "*.pth"))
for file in model_files:
    model_name = os.path.basename(file)
    model = SitelenPonaCNN(num_classes=len(GLYPHS)).to(device)
    model.load_state_dict(torch.load(file, map_location=device))
    model.eval() # CRITICAL: Sets model to evaluation mode (turns off dropout)
    loaded_models[model_name] = model

print(f"Loaded {len(loaded_models)} models for testing.")

Loaded 3 models for testing.


In [ ]:
# --- PREPROCESS
def format_for_model(pil_img):
    """Converts a raw drawing into a perfect 28x28 normalized tensor."""
    img = pil_img.convert('L')
    arr = np.array(img)
    
    # invert colors if necessary (white lines on black background is how model was trained)
    if np.mean(arr) > 127:
        arr = 255 - arr
        
    # crop
    img_inverted = Image.fromarray(arr)
    bbox = img_inverted.getbbox()
    
    if not bbox:
        return torch.zeros((1, 1, IMG_SIZE, IMG_SIZE)).to(device)
        
    img_cropped = img_inverted.crop(bbox)
    
    # normalize and center in 28x28
    w, h = img_cropped.size
    max_dim = 24 
    scale = max_dim / max(w, h)
    new_w = max(1, int(w * scale))
    new_h = max(1, int(h * scale))
    
    img_resized = img_cropped.resize((new_w, new_h), Image.Resampling.BILINEAR)
    
    final_img = Image.new('L', (IMG_SIZE, IMG_SIZE), color=0)
    offset_x = (IMG_SIZE - new_w) // 2
    offset_y = (IMG_SIZE - new_h) // 2
    final_img.paste(img_resized, (offset_x, offset_y))
    
    tensor = torch.tensor(np.array(final_img), dtype=torch.float32)
    tensor = tensor.unsqueeze(0).unsqueeze(0) / 255.0
    
    return tensor.to(device)

# --- INFERENCE
def predict_drawing(drawing):
    if drawing is None or len(loaded_models) == 0:
        return "No drawing or no models loaded."
    
    if isinstance(drawing, dict):
        img = drawing['composite']
    else:
        img = drawing

    tensor = format_for_model(img)
    
    results = []
    
    # compare models
    with torch.no_grad():
        for name, model in loaded_models.items():
            outputs = model(tensor)
            
            probabilities = F.softmax(outputs, dim=1)[0]
            
            confidence, predicted_idx = torch.max(probabilities, 0)
            
            glyph_name = GLYPHS[predicted_idx.item()]
            conf_pct = confidence.item() * 100
            
            results.append(f"model: {name}:\n=> {glyph_name} ({conf_pct:.1f}% confidence)\n")
            
    return "\n".join(results)

In [ ]:
# --- sketchpad
demo = gr.Interface(
    fn=predict_drawing,
    inputs=gr.Sketchpad(type="pil", label="Draw your sitelen pona here!"),
    outputs=gr.Textbox(label="Model Predictions", lines=10),
    title="Sitelen Pona Multi-Model Inference",
    description="Draw a glyph. The script will format it to 28x28 and pass it to all models in the ./models folder."
)

demo.launch(inline=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
